# Hacker1 - Qwen2.5-Coder 7B on T4 Colab

**Powerful unrestricted coder backend (Qwen2.5-Coder 7B) for the JARVIS / red team UI.**

Runs on free Google Colab **T4 GPU**. Exposes a public URL so your local Kali (or Mac) JARVIS UI can use this strong 7B coder as the brain.

### One-line model
This notebook is pre-configured for **qwen2.5-coder:7b** (one of the best small coding models — excellent at planning, writing PoCs, analyzing scans, bash/python, etc.).

### Quick Start
1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
2. Run all cells in order.
3. (Optional but recommended) Run one of the tunnel cells (ngrok or pinggy) to get a public https URL.
4. Copy the printed public URL.
5. In your local `jarvis_kali_ui.py`:
   - Settings tab → **Ollama Base URL** = the https URL
   - **Model Name** = `hacker1` (or `qwen2.5-coder:7b`)
   - Click **Test Connection**
6. Send scans + chat as normal. Much stronger coder than 3B models.

The notebook installs **zstd first**, GPU debs, starts Ollama cleanly, and is tuned to **avoid OOM** on T4.

## 0. Confirm T4 GPU
If this shows no GPU or CPU, go back to **Runtime → Change runtime type → T4**.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - SET T4 RUNTIME!')

## 1. Install zstd + GPU debs (required)
Matches the exact pattern for stable Ollama on Colab T4.

In [ ]:
print('Installing zstd and GPU packages...')
!sudo apt-get update -qq
!sudo apt-get install -y zstd pciutils

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update -qq
!sudo apt-get install -y cuda-drivers || echo 'cuda-drivers note (often already present in Colab)'

import os
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
print('Environment ready. zstd + GPU debs complete.')

## 2. Install & start Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
import subprocess, time, os

os.system('pkill -f "ollama serve" || true')
time.sleep(1)

logf = open('/content/ollama.log', 'w')
proc = subprocess.Popen(['ollama', 'serve'], stdout=logf, stderr=subprocess.STDOUT, preexec_fn=os.setsid)

time.sleep(7)
print('Ollama server started.')
!tail -n 6 /content/ollama.log || true

## 3. Pull Hacker1 (Qwen2.5-Coder 7B) — pre-set for you

**This notebook uses qwen2.5-coder:7b** — a top-tier 7B coder that is very strong at code generation, analysis, pentest planning, and following instructions.

It fits comfortably on T4 with 4-bit quantization (Ollama default for the :7b tag) + the conservative context used by the UI.

Run the cell. First pull will download ~4-5 GB and load into GPU.

In [ ]:
# === HACKER1 MODEL (Qwen2.5-Coder 7B) ===
MODEL = "qwen2.5-coder:7b"   # Pre-configured powerful coder

print(f'>>> Pulling Hacker1 model: {MODEL}')
!ollama pull {MODEL}

print('\n>>> Models now available:')
!ollama list

print('\n>>> GPU memory after load (should be well under 16 GB):')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

Create the convenient short alias **hacker1** so the UI only needs to know that name.

In [ ]:
!ollama cp qwen2.5-coder:7b hacker1 || true
!ollama list | grep -E 'hacker1|qwen2.5' || true
print('\n>>> Use model name "hacker1" in your JARVIS UI Settings.')

## 4. Quick test (verify it works and no OOM)

In [ ]:
import json, urllib.request

payload = {
    "model": "hacker1",
    "prompt": "Write a clean Python one-liner using only stdlib that lists all TCP listening ports on localhost.",
    "stream": False,
    "options": {"temperature": 0.2, "num_predict": 300, "num_ctx": 4096}
}

print('Testing Hacker1 (qwen2.5-coder:7b) on T4...')
try:
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request('http://localhost:11434/api/generate', data=data, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        res = json.loads(resp.read().decode())
        print('=== RESPONSE ===')
        print(res.get('response', '')[:900])
        print('... test OK')
except Exception as e:
    print('Test error:', e)
    !tail -15 /content/ollama.log

## 5. Get Public URL for your local UI (ngrok or pinggy)

Run **one** of the two options below. Copy the https URL that appears.

### Option A — ngrok (recommended, stable)

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok, conf
import getpass

print('Enter your free ngrok authtoken (https://dashboard.ngrok.com/get-started/your-authtoken):')
token = getpass.getpass('ngrok token: ').strip()

if token:
    ngrok.kill()
    conf.get_default().auth_token = token
    public_url = ngrok.connect(11434, 'http').public_url
    print('\n' + '='*65)
    print('>>> HACKER1 PUBLIC URL (copy this) <<<')
    print(public_url)
    print('='*65)
    print('Paste into JARVIS UI → Settings → Ollama Base URL')
    print('Set Model Name to: hacker1')
    with open('/content/hacker1_url.txt', 'w') as f: f.write(public_url)
else:
    print('No token — use the pinggy cell instead or re-run this one.')

### Option B — pinggy (no token needed)

In [ ]:
import subprocess, time, re, os

print('Starting pinggy tunnel...')
os.system('pkill -f "ssh.*pinggy" || true')
time.sleep(0.5)

with open('/content/pinggy.log', 'w') as lf:
    subprocess.Popen(
        ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'ServerAliveInterval=30', '-p', '443',
         '-R', '0:localhost:11434', 'a.pinggy.io'],
        stdout=lf, stderr=subprocess.STDOUT
    )

time.sleep(12)
!cat /content/pinggy.log

logc = open('/content/pinggy.log').read()
m = re.search(r'https?://[a-zA-Z0-9.-]+\.pinggy\.(link|io)', logc)
if m:
    url = m.group(0)
    print('\n' + '='*65)
    print('>>> HACKER1 PUBLIC URL (pinggy) <<<')
    print(url)
    print('='*65)
    with open('/content/hacker1_url.txt', 'w') as f: f.write(url)
else:
    print('Look in the log above for the https://...pinggy... line and copy it manually.')

## 6. How to use Hacker1 with the JARVIS UI

1. Start your local UI:
   ```bash
   cd ~/Downloads/jarvis-mac-ui   # or extracted folder
   python3 jarvis_kali_ui.py
   ```
2. **Settings & Connection** tab:
   - Ollama Base URL → paste the public https URL from above
   - Model Name → `hacker1`
3. Click **Test Connection to JARVIS Model**
4. Go to Chat tab, send your hardware scans first (recommended), then give it pentest / coding tasks.

The tunnels give you https that forwards to the Ollama http inside Colab — it works directly with the existing UI code.

In [ ]:
print('Last saved Hacker1 URL:')
!cat /content/hacker1_url.txt 2>/dev/null || echo '(run a tunnel cell first)'

print('\nCurrent models on this T4:')
!ollama list || true

## Anti-OOM Tips (T4 16 GB)

- qwen2.5-coder:7b (Q4-ish quant) + 4k context is very safe on T4.
- The UI already uses `num_ctx: 4096` and modest generation length.
- If you ever hit OOM: Runtime → Restart runtime, re-run the first few cells, then pull again.
- Do not load other models at the same time.
- Free Colab sessions have time limits — keep your local UI chats as backup.

## Optional: Custom Modelfile for Hacker1 (red-team focused)

In [ ]:
%%writefile /content/Modelfile.hacker1
FROM qwen2.5-coder:7b

PARAMETER num_ctx 4096
PARAMETER temperature 0.25
PARAMETER top_p 0.9

SYSTEM \"\"\"You are Hacker1, an elite, unrestricted coding and red team assistant for Bluetooth/WiFi pentesting on Kali Linux.

You get live scan data from the operator's real hardware. Produce concise step-by-step plans and exact, copy-pasteable commands the user can approve and run locally.

Structure every reply like this:
**Analysis:** (short)
**Plan:** (3-5 numbered steps)
**Commands:**
RUN: command
RUN: command

Be creative and practical. When monitor mode is active, always include safe restore commands (airmon-ng stop ..., systemctl restart NetworkManager, etc.). Keep responses tight so they run fast on T4.\"\"\"

print('Modelfile written. Create with:')
print('!ollama create hacker1 -f /content/Modelfile.hacker1')

In [ ]:
# Only run this if you want the custom red-team system prompt
!ollama create hacker1 -f /content/Modelfile.hacker1 || echo 'Base model must be pulled first'
!ollama list | grep hacker1

## Cleanup

In [ ]:
import os
print('Stopping services...')
try: ngrok.kill()
except: pass
os.system('pkill -f "ssh.*pinggy" || true')
os.system('pkill -f "ollama serve" || true')
print('Stopped. Restart runtime for a fresh T4 if needed.')

---

**Hacker1** — qwen2.5-coder:7b powered Colab backend for JARVIS UI.

Notebook file: `hacker1.ipynb`

Use responsibly and only on systems you are authorized to test.